# Define Recipe

In [ ]:
from model import SimpleNetwork

from nvflare.app_opt.tf.recipes.fedavg import FedAvgRecipe
from nvflare.recipe.sim_env import SimEnv
from nvflare.recipe.prod_env import ProdEnv

recipe = FedAvgRecipe(
    name="hello-tf-mlflow",
    min_clients=1,
    num_rounds=3,
    initial_model=SimpleNetwork(),
    train_script="client.py",
    train_args=f"--batch_size 32",
    mlflow_tracking_uri="https://rrayrz6j-nvflmlflowserver.xenon.lepton.run"
)

recipe.export("transfer")

# Run in Simulation Environment

In [75]:
env = SimEnv(num_clients=1, num_threads=1)
recipe.execute(env=env)

2025-08-29 15:37:31,292 - INFO - model selection weights control: {}
2025-08-29 15:37:32,660 - INFO - Tensorboard records can be found in /tmp/nvflare/simulation/hello-pt-mlflow/server/simulate_job/tb_events you can view it using `tensorboard --logdir=/tmp/nvflare/simulation/hello-pt-mlflow/server/simulate_job/tb_events`
2025-08-29 15:37:32,660 - INFO - Initializing MLflow tracking for sites: ['site-1']
2025-08-29 15:37:33,276 - INFO - Experiment=<Experiment: artifact_location='mlflow-artifacts:/902671310698136517', creation_time=1756401013517, experiment_id='902671310698136517', last_update_time=1756401013517, lifecycle_stage='active', name='nvflare-poc-lepton-fl-experiment', tags={}>
2025-08-29 15:37:33,658 - INFO - MLflow tracking initialization completed successfully
2025-08-29 15:37:33,659 - INFO - Initializing ScatterAndGather workflow for Federated Averaging.
2025-08-29 15:37:33,659 - INFO - Both source_ckpt_file_full_name and ckpt_preload_path are not provided. Using the defaul

# Run in Production Environment

In [ ]:
env = ProdEnv(startup_kit_dir=".")
recipe.execute(env=env)


# Monitoring

In [ ]:
from nvflare.fuel.flare_api.flare_api import new_secure_session
 
username = "admin@nvidia.com"
sess = new_secure_session(
    username=username,
    startup_kit_location=".",
    timeout=30
)
print(sess.get_system_info())

job_id = "911afd39-4be1-4a5a-bdee-4291854c4110"

## List jobs

In [ ]:
sess.list_jobs()

In [ ]:
from nvflare.fuel.flare_api.flare_api import Session

def sample_cb(
        session: Session, job_id: str, job_meta, *cb_args, **cb_kwargs
    ) -> bool:
    """
    Custom callback function for job monitoring.
    
    Args:
        session: The FLARE session
        job_id: The ID of the job being monitored
        job_meta: Metadata about the job's current state
        cb_args: Additional positional arguments
        cb_kwargs: Additional keyword arguments
        
    Returns:
        bool: True to continue monitoring, False to stop
    """
    if job_meta["status"] == "RUNNING":
        if cb_kwargs["cb_run_counter"]["count"] < 1:
            print(job_meta)
            print(cb_kwargs["cb_run_counter"])
        else:
            print(".", end="")
    else:
        print("\n" + str(job_meta))
    
    cb_kwargs["cb_run_counter"]["count"] += 1
    return True

# Monitor the job with our custom callback
print(f"Monitoring job {job_id}...")
#result = sess.monitor_job(job_id, cb=sample_cb, cb_run_counter={"count":0})
result = sess.monitor_job(job_id)
print(f"\nMonitoring completed with result: {result}")

### (Optionally) Abort job

In [ ]:
sess.abort_job(job_id)